In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        if member.name == "NCS" or member.name.startswith("NCS/"):
            member.name = "OmniAligner" + member.name[3:]
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "OmniPred"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%cd {PACKAGE_ROOT}


## Download and run
Use a fresh Python 3.10/3.11 Jupyter kernel with `wget` and `git`. The first two commands retrieve the revision package and repository without private paths or credentials. This notebook reconstructs tables from frozen scoring records; it does not rerun predictor training or inference.


# OmniPred revision benchmark: reproduce the submitted tables

Rebuild the fixed S1/S2 seven-method cell-RNA comparison, iSTAR regional comparison, Path2Space native-tile comparison and CRC GigaTIME protein-region comparison. These tasks remain separate; there is no combined ranking.

This notebook recomputes summaries from the preserved gene-, seed- and marker-level scoring records. It does not retrain predictors or rerun inference. The historical frozen OmniPred condition in the primary tables is not replaced with the later repaired-triplet development model. S1/S2 are from one block; S2 was previously inspected. GHIST full-cohort scores include fallback cells.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "manifest.json").is_file() and (p / "data/scores_by_gene.csv").is_file())
DATA = ROOT / "data"
OUTPUT = ROOT / "results"
OUTPUT.mkdir(exist_ok=True)
manifest = json.loads((ROOT / "manifest.json").read_text())
for rel, digest in manifest.items():
    assert hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() == digest, rel
validation = {}
ATOL = 1e-10


## 1. Cell-level RNA: seven methods on the same 313 genes
Gene-wise PCC/Spearman and normalized errors are averaged over all 313 genes, then across the declared seeds. Median and IQR are calculated per seed and then averaged. Within-cell profile PCC and cosine are retained from the sealed per-seed summaries; they cannot be recovered from gene-wise correlations. Constant correlations retain the original zero-score convention and fixed denominator. Seed SD uses ddof=1; a single model has no estimated seed SD.

In [2]:
genes = pd.read_csv(DATA / "scores_by_gene.csv")
seeds = pd.read_csv(DATA / "scores_by_seed.csv")
reference = pd.read_csv(DATA / "FIG4_AGGREGATE_SCORES.csv")
methods = reference.method.drop_duplicates().tolist()
keys = ["method", "seed", "split", "scope"]
g = genes[(genes.scope == "full_cohort") & genes.method.isin(methods)].copy()
s = seeds[(seeds.scope == "full_cohort") & seeds.method.isin(methods)].copy()
assert not g.duplicated(keys + ["gene"]).any()
assert g.groupby(keys).size().eq(313).all()
rows = []
for key, frame in g.groupby(keys, sort=False):
    row = dict(zip(keys, key))
    row.update(gene_PCC=frame.PCC.mean(), gene_PCC_median=frame.PCC.median(),
               gene_PCC_Q1=frame.PCC.quantile(.25), gene_PCC_Q3=frame.PCC.quantile(.75),
               gene_PCC_IQR=frame.PCC.quantile(.75)-frame.PCC.quantile(.25),
               gene_Spearman=frame.Spearman.mean(), NMSE=frame.NMSE.mean(), NMAE=frame.NMAE.mean())
    rows.append(row)
per_seed = pd.DataFrame(rows)
metrics = ["gene_PCC", "gene_PCC_median", "gene_Spearman", "NMSE", "NMAE"]
check = per_seed.merge(s, on=keys, suffixes=("_rebuilt", "_saved"), validate="one_to_one")
for metric in metrics:
    np.testing.assert_allclose(check[metric+"_rebuilt"], check[metric+"_saved"], atol=ATOL, rtol=0)
per_seed = per_seed.merge(s[keys + ["profile_PCC", "CosSim", "native_covered", "fallback"]], on=keys, validate="one_to_one")
means = per_seed.groupby(["method", "split"], sort=False).mean(numeric_only=True).reset_index()
shared = means.merge(reference, on=["method", "split"], suffixes=("_rebuilt", "_reference"), validate="one_to_one")
metrics += ["profile_PCC", "CosSim", "native_covered", "fallback"]
for metric in metrics:
    np.testing.assert_allclose(shared[metric+"_rebuilt"], shared[metric+"_reference"], atol=ATOL, rtol=0)
for split in ["S1", "S2"]:
    display(Markdown(f"### {split}: cell RNA"))
    table = means[means.split.eq(split)].set_index("method").reindex(methods)
    display(table[["gene_PCC", "gene_PCC_median", "gene_PCC_IQR", "gene_Spearman", "profile_PCC", "CosSim", "NMSE", "NMAE", "fallback"]].round(6))
means.to_csv(OUTPUT / "cell_RNA_tables.csv", index=False)
per_seed.to_csv(OUTPUT / "cell_RNA_per_seed.csv", index=False)
validation["cell_RNA"] = {"rows": len(shared), "metrics_compared": len(metrics)*len(shared), "match": True}


### S1: cell RNA

,gene_PCC,gene_PCC_median,gene_PCC_IQR,gene_Spearman,profile_PCC,CosSim,NMSE,NMAE,fallback
method,,,,,,,,,
OmniPred_selected,0.275905,0.255634,0.255808,0.260558,0.678042,0.739156,0.849830,0.473577,0.0
SpatialEx,0.227407,0.195371,0.250906,0.222546,0.654990,0.720067,0.879310,0.460826,0.0
GHIST,0.222852,0.194711,0.385851,0.212696,0.665856,0.728158,0.869072,0.429612,1.0
CONCH_Linear,0.167984,0.110403,0.222750,0.166980,0.591491,0.668084,1.486951,0.810930,0.0
CONCH_MLP_512,0.259660,0.236099,0.248084,0.239351,0.670517,0.733330,0.859573,0.527654,0.0
CONCH_Ridge,0.257980,0.232797,0.223798,0.247397,0.669307,0.732267,0.846657,0.513203,0.0
train_mean,0.000000,0.000000,0.000000,0.000000,0.554578,0.644654,0.939079,0.572202,0.0


### S2: cell RNA

,gene_PCC,gene_PCC_median,gene_PCC_IQR,gene_Spearman,profile_PCC,CosSim,NMSE,NMAE,fallback
method,,,,,,,,,
OmniPred_selected,0.269027,0.241350,0.246761,0.257285,0.631561,0.703194,0.969832,0.507441,0.0
SpatialEx,0.220592,0.180938,0.264922,0.212380,0.613115,0.687175,1.003302,0.494768,0.0
GHIST,0.225952,0.195137,0.350907,0.216557,0.621713,0.694863,0.987333,0.466342,1140.0
CONCH_Linear,0.156635,0.104328,0.224763,0.158922,0.536399,0.623785,1.760087,0.866480,0.0
CONCH_MLP_512,0.249927,0.222452,0.255981,0.233216,0.623954,0.697852,0.978273,0.558197,0.0
CONCH_Ridge,0.252296,0.225893,0.230172,0.242691,0.620766,0.695957,0.965032,0.540775,0.0
train_mean,0.000000,0.000000,0.000000,0.000000,0.500543,0.606905,1.058090,0.594672,0.0


## 2. Marker recovery
PGR and MKI67 correlations are shown separately. GHIST's full-cohort PGR includes fallback cells; inspect its native-supported rows below. Sparse detection does not by itself establish the cause of a low PCC. Gene-distribution IQR is not a patient confidence interval.

In [3]:
markers = genes[genes.method.isin(methods) & genes.gene.isin(["PGR", "MKI67"])].groupby(["method", "split", "scope", "gene"], sort=False).agg(PCC=("PCC", "mean"), seeds=("seed", "nunique"), constant_seeds=("prediction_constant", "sum")).reset_index()
display(markers[markers.split.eq("S2")])
markers.to_csv(OUTPUT / "marker_PCC.csv", index=False)


,method,split,scope,gene,PCC,seeds,constant_seeds
2,CONCH_MLP_512,S2,full_cohort,MKI67,0.139108,3,0
3,CONCH_MLP_512,S2,full_cohort,PGR,0.015148,3,0
6,CONCH_Linear,S2,full_cohort,MKI67,0.070241,3,0
7,CONCH_Linear,S2,full_cohort,PGR,0.033103,3,0
10,OmniPred_selected,S2,full_cohort,MKI67,0.163966,1,0
11,OmniPred_selected,S2,full_cohort,PGR,0.090830,1,0
14,SpatialEx,S2,full_cohort,MKI67,0.131327,3,0
15,SpatialEx,S2,full_cohort,PGR,-0.035698,3,0
20,GHIST,S2,full_cohort,MKI67,0.040014,3,0
21,GHIST,S2,full_cohort,PGR,0.191050,3,0


## 3. iSTAR regions and Path2Space native tiles
The per-gene records already encode the fixed shared supports and native-field endpoint. Aggregate only those exact supports. iSTAR uses 313 genes in 55/100-µm circles; Path2Space's primary smoothed tile endpoint has 239 mapped genes. Unavailable genes stay unavailable. These correlations are not cell-level accuracy. The same historical selected OmniPred field condition is retained from the archived matched-support comparison.

In [4]:
regional = pd.read_csv(DATA / "regional/per_gene.csv")
references = pd.concat([pd.read_csv(DATA / "regional/comparison_istar.csv"), pd.read_csv(DATA / "regional/comparison_path2.csv")], ignore_index=True)
keys = ["method", "cohort", "diameter_um", "endpoint"]
regional_rows=[]
for _, expected in references.iterrows():
    sub = regional.copy()
    for key in keys:
        sub = sub[sub[key].eq(expected[key])]
    # The native mapped panel is the jointly estimable support, not all 313 requested genes.
    sub = sub[sub.OmniPred_PCC.notna() & sub.comparator_PCC.notna()]
    assert len(sub) == int(expected.genes), (expected[keys].tolist(), len(sub), expected.genes)
    row = {key:expected[key] for key in keys}
    row.update(genes=len(sub), regions=int(expected.regions), contributor_cells=int(expected.contributor_cells))
    for col in ["OmniPred_PCC", "comparator_PCC", "delta"]:
        row[col] = sub[col].mean()
        np.testing.assert_allclose(row[col],expected[col],atol=ATOL,rtol=0)
    regional_rows.append(row)
rebuilt_regions = pd.DataFrame(regional_rows)
display(Markdown("### iSTAR: regional RNA"))
display(rebuilt_regions[rebuilt_regions.method.eq("iSTAR")])
display(Markdown("### Path2Space: native tiles (primary)"))
display(rebuilt_regions[rebuilt_regions.endpoint.eq("smoothed_primary")])
rebuilt_regions.to_csv(OUTPUT / "regional_tile_tables.csv", index=False)
validation["regional_tile"]={"rows":len(rebuilt_regions),"PCC_and_delta_values":len(rebuilt_regions)*3,"match":True}


### iSTAR: regional RNA

,method,cohort,diameter_um,endpoint,genes,regions,contributor_cells,OmniPred_PCC,comparator_PCC,delta
0,iSTAR,S1,55,native,313,402,941,0.379043,0.224876,0.154167
1,iSTAR,S1,100,native,313,576,3158,0.454897,0.323679,0.131218
2,iSTAR,S2,55,native,313,1868,4549,0.372719,0.249766,0.122953
3,iSTAR,S2,100,native,313,2556,14843,0.454382,0.370636,0.083746


### Path2Space: native tiles (primary)

,method,cohort,diameter_um,endpoint,genes,regions,contributor_cells,OmniPred_PCC,comparator_PCC,delta
4,Path2,S1,0,smoothed_primary,239,527,3145,0.481958,0.078565,0.403393
6,Path2,S2,0,smoothed_primary,239,1867,13870,0.479203,0.117008,0.362195


## 4. CRC protein regions: GigaTIME and stored OmniPred fields
Macro PCC is the mean over nine common markers on 178 shared regions. Hard-area GigaTIME is primary; probability means are secondary. Stored OmniPred fusion/ensemble H&E-only fields retain their stated output scale. Differing units preclude a calibrated NMSE ranking. Additional archived field variants remain visible and are not silently relabelled as the selected model.

In [5]:
protein = pd.read_csv(DATA / "crc/common9_per_marker.csv")
protein_reference = pd.read_csv(DATA / "crc/common9_comparison.csv")
assert protein.groupby("model").size().eq(9).all()
assert protein.shared_regions.eq(178).all()
protein_rebuilt = protein.groupby("model", sort=False).agg(common9_macro_PCC=("region_PCC", "mean"), markers=("marker", "nunique"), shared_regions=("shared_regions", "first"), output_unit=("output_unit", "first")).reset_index()
check = protein_rebuilt.merge(protein_reference,on="model",suffixes=("_rebuilt","_reference"),validate="one_to_one")
np.testing.assert_allclose(check.common9_macro_PCC_rebuilt,check.common9_macro_PCC_reference,atol=ATOL,rtol=0)
display(protein_rebuilt)
protein_rebuilt.to_csv(OUTPUT / "CRC_protein_regions.csv", index=False)
validation["CRC_protein"]={"models":len(check),"markers_per_model":9,"regions":178,"match":True}
validation["scope"]="Reaggregation of frozen scoring records; no training or prediction rerun"
(OUTPUT / "VALIDATION.json").write_text(json.dumps(validation,indent=2))
print(json.dumps(validation,indent=2))


,model,common9_macro_PCC,markers,shared_regions,output_unit
0,GigaTIME_official_native_GPU_batch8_hard_primary,0.097031,9,178,thresholded pixel area fraction
1,GigaTIME_official_native_GPU_batch8_soft_secon...,0.094829,9,178,mean probability
2,fusion_HE_only,0.755885,9,178,existing stored model output; original conditi...
3,fusion_global_residual,0.755885,9,178,existing stored model output; original conditi...
4,fusion_feature_residual,0.806016,9,178,existing stored model output; original conditi...
5,fusion_alignedXY_residual,0.837553,9,178,existing stored model output; original conditi...
6,fusion_geometry_residual,0.853571,9,178,existing stored model output; original conditi...
7,ensemble_HE_only,0.756156,9,178,existing stored model output; original conditi...
8,ensemble_global_residual,0.756156,9,178,existing stored model output; original conditi...
9,ensemble_feature_residual,0.803771,9,178,existing stored model output; original conditi...


{
  "cell_RNA": {
    "rows": 14,
    "metrics_compared": 126,
    "match": true
  },
  "regional_tile": {
    "rows": 8,
    "PCC_and_delta_values": 24,
    "match": true
  },
  "CRC_protein": {
    "models": 47,
    "markers_per_model": 9,
    "regions": 178,
    "match": true
  },
  "scope": "Reaggregation of frozen scoring records; no training or prediction rerun"
}
